In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import types as t
from pyspark.sql.window import Window
from datetime import datetime


## Função Para Ler a Partição

In [0]:
def ler_ultima_particao_delta(spark, base_path):
  """
  Essa fução é para ler a ultima partição dos volumes delta baseada na coluna 'data_processamento'
  """
  try: 
      # Descobrir as partições direto no storage 
      particoes = dbutils.fs.ls(base_path)
      datas = [
              int(p.name.split('=')[1].replace('/', '')) 
              for p in particoes if "data_processamento=" in p.name
      ]
      
      if not datas:
          print(f"Nenhuma partição encontrada em {base_path}")
          return None
      else:
          ultima_particao = max(datas)
          print(f"[{base_path}] Ultima partição: {ultima_particao}")
          return spark.read.format("delta").load(f"{base_path}/data_processamento={ultima_particao}")
  except Exception as e:
    print(f"Erro ao ler caminho {base_path}: {e}")
    return None

## CVM - Fundos Investimentos - Registros Classes

In [0]:
bronze_path_registro_classe_cvm = "/Volumes/workspace/case_spark_cvm/bronze/registro_classe_cvm/"

df_registro_classe_cvm = ler_ultima_particao_delta(spark, bronze_path_registro_classe_cvm)

### 1.1 tratemento silver

#### 1.1.1 Retirando dados nulos de Colunas Cores

In [0]:
# Lista de colunas de caso falte dados precisamos dropa
colunas_obrigatorias = ['ID_Registro_Fundo', 'ID_Registro_Classe', 'CNPJ_Classe', 'Tipo_Classe', 'Data_Inicio','Situacao']

# Aplicando a filtro para dropar as colunas
df_registro_classe_cvm = df_registro_classe_cvm.dropna(subset=colunas_obrigatorias)# Lista de colunas de caso falte dados precisamos dropa
colunas_obrigatorias = ['ID_Registro_Fundo', 'ID_Registro_Classe', 'CNPJ_Classe', 'Tipo_Classe', 'Data_Inicio','Situacao']

# Aplicando a filtro para dropar as colunas
df_registro_classe_cvm = df_registro_classe_cvm.dropna(subset=colunas_obrigatorias)

#### 1.1.2 Retirando dados duplicados

Nesta base podem existir registros com ***CNPJ_Classe*** duplicados. Para tratar essa situação, criamos uma coluna de ***prioridade_situacao***, essa coluna irar criar uma regra onde se a situação estiver **"Em Funcionamento Normal"**, vai ter prioridade em seguida mantemos apenas o registro mais recente, considerando o campo ***Data_Registro***.

In [0]:
df_registro_classe_cvm = df_registro_classe_cvm\
    .withColumn("prioridade_situacao", f.when(f.col("Situacao") == "Em Funcionamento Normal", 1).otherwise(2))

window_spec_registros_classe = Window.partitionBy("CNPJ_Classe").orderBy(f.col("prioridade_situacao"), f.col("Data_Registro").desc())

df_registro_classe_cvm = df_registro_classe_cvm.withColumn("row_num", f.row_number().over(window_spec_registros_classe))

df_registro_classe_cvm = df_registro_classe_cvm.filter(f.col("row_num") == 1).drop("row_num", "prioridade_situacao")

#### 1.1.3 Tratamento do Tipo de Dado

In [0]:
# Dropando a Data de Processamento da Bronze 
df_registro_classe_cvm = df_registro_classe_cvm.drop("data_processamento")

# Criando a Data de Processamento da silver
df_registro_classe_cvm = df_registro_classe_cvm.withColumn(
    "data_processamento",
    f.date_format(f.current_date(), "yyyyMMdd").cast("int")
)

In [0]:

df_registro_classe_cvm = df_registro_classe_cvm \
    .withColumn('id_registro_fundo', f.col('ID_Registro_Fundo').cast(t.IntegerType())) \
    .withColumn('id_registro_classe', f.col('ID_Registro_Classe').cast(t.IntegerType())) \
    .withColumn('cnpj_classe', f.col('CNPJ_Classe').cast(t.StringType())) \
    .withColumn('codigo_cvm', f.col('Codigo_CVM').cast(t.IntegerType())) \
    .withColumn('data_registro', f.col('Data_Registro').cast(t.DateType())) \
    .withColumn('data_constituicao', f.col('Data_Constituicao').cast(t.DateType())) \
    .withColumn('data_inicio', f.col('Data_Inicio').cast(t.DateType())) \
    .withColumn('tipo_classe', f.col('Tipo_Classe').cast(t.StringType())) \
    .withColumn('denominacao_social', f.col('Denominacao_Social').cast(t.StringType())) \
    .withColumn('situacao', f.col('Situacao').cast(t.StringType())) \
    .withColumn('data_inicio_situacao', f.col('Data_Inicio_Situacao').cast(t.DateType())) \
    .withColumn('classificacao', f.col('Classificacao').cast(t.StringType())) \
    .withColumn('indicador_desempenho', f.col('Indicador_Desempenho').cast(t.StringType())) \
    .withColumn('classe_cotas', f.col('Classe_Cotas').cast(t.StringType())) \
    .withColumn('classificacao_anbima', f.col('Classificacao_Anbima').cast(t.StringType())) \
    .withColumn('tributacao_longo_prazo', f.col('Tributacao_Longo_Prazo').cast(t.StringType())) \
    .withColumn('entidade_investimento', f.col('Entidade_Investimento').cast(t.StringType())) \
    .withColumn('permitido_aplicacao_cemporcento_exterior', f.col('Permitido_Aplicacao_CemPorCento_Exterior').cast(t.StringType())) \
    .withColumn('classe_esg', f.col('Classe_ESG').cast(t.StringType())) \
    .withColumn('forma_condominio', f.col('Forma_Condominio').cast(t.StringType())) \
    .withColumn('exclusivo', f.col('Exclusivo').cast(t.StringType())) \
    .withColumn('publico_alvo', f.col('Publico_Alvo').cast(t.StringType())) \
    .withColumn('patrimonio_liquido', f.col('Patrimonio_Liquido').cast(t.DecimalType(25,2))) \
    .withColumn('data_patrimonio_liquido', f.col('Data_Patrimonio_Liquido').cast(t.DateType())) \
    .withColumn('cnpj_auditor', f.col('CNPJ_Auditor').cast(t.StringType())) \
    .withColumn('auditor', f.col('Auditor').cast(t.StringType())) \
    .withColumn('cnpj_custodiante', f.col('CNPJ_Custodiante').cast(t.StringType())) \
    .withColumn('custodiante', f.col('Custodiante').cast(t.StringType())) \
    .withColumn('cnpj_controlador', f.col('CNPJ_Controlador').cast(t.StringType())) \
    .withColumn('controlador', f.col('Controlador').cast(t.StringType()))\
    .withColumn('data_processamento', f.col('data_processamento').cast(t.IntegerType()))


### 1.2 Salvar na camada Silver

In [0]:
data_proc = int(datetime.now().strftime(f"%Y%m%d"))

df_registro_classe_cvm.write \
    .mode('overwrite') \
    .partitionBy("data_processamento") \
    .option("replaceWhere", f"data_processamento = {data_proc}")\
    .format('delta')\
    .saveAsTable("workspace.case_spark_cvm.silver_registro_classe_cvm")